# DreamerV3 HM3D Easy — Post-Training Analysis

Analyze the trained DreamerV3 model after the 5M-step run on HM3D easy episodes.

**Training config:**
- 64×64 obs, geodesic < 5m, max_episode_steps=200
- 5M steps on `gpu_h100`, checkpoints every 500k
- Split: train, reward: geodesic delta + 10.0 success bonus

**This notebook:**
1. Plot training curves (from wandb or SLURM logs)
2. Run deterministic eval on val split
3. Analyze SR/SPL by category and difficulty
4. Compare to random baseline

In [ ]:
import os, json, glob
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

project_root = Path(os.path.abspath("")).parent if Path(os.path.abspath("")).name == "notebooks" else Path(os.path.abspath(""))
os.chdir(project_root)

# Find the latest training run
RUN_DIR = None
candidates = sorted(glob.glob("output/dreamerv3-easy/run-*"))
if candidates:
    RUN_DIR = candidates[-1]
    print(f"Latest run: {RUN_DIR}")
    for f in sorted(Path(RUN_DIR).glob("*")):
        print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")
else:
    print("No training runs found in output/dreamerv3-easy/")
    print("Run: sbatch scripts/slurm/train_dreamer_easy.sbatch")

In [ ]:
# Load training metrics from wandb (if available)
try:
    import wandb
    api = wandb.Api()
    runs = api.runs("dreamerv3-objectnav", filters={"config.max_geodesic": 5.0})
    if runs:
        run = runs[0]
        history = run.history(samples=5000)
        print(f"Loaded wandb run: {run.name} ({len(history)} samples)")
        HAS_WANDB = True
    else:
        print("No matching wandb runs found")
        HAS_WANDB = False
except Exception as e:
    print(f"wandb not available: {e}")
    HAS_WANDB = False

# Fallback: parse SLURM output logs
if not HAS_WANDB and RUN_DIR:
    slurm_logs = sorted(glob.glob("output/dreamerv3-easy/slurm-*.out"))
    if slurm_logs:
        print(f"Parsing SLURM log: {slurm_logs[-1]}")
        steps_log, wm_log, actor_log, critic_log = [], [], [], []
        ep_steps, ep_rewards, ep_success = [], [], []
        with open(slurm_logs[-1]) as f:
            for line in f:
                if line.startswith("[step") and "wm=" in line:
                    parts = line.strip().split()
                    step = int(parts[1].rstrip("]"))
                    wm = float([p for p in parts if p.startswith("wm=")][0].split("=")[1])
                    actor = float([p for p in parts if p.startswith("actor=")][0].split("=")[1])
                    critic = float([p for p in parts if p.startswith("critic=")][0].split("=")[1])
                    steps_log.append(step)
                    wm_log.append(wm)
                    actor_log.append(actor)
                    critic_log.append(critic)
                elif line.startswith("[step") and "reward=" in line:
                    parts = line.strip().split()
                    step = int(parts[1].rstrip("]"))
                    reward = float([p for p in parts if p.startswith("reward=")][0].split("=")[1])
                    success = float([p for p in parts if p.startswith("success=")][0].split("=")[1])
                    ep_steps.append(step)
                    ep_rewards.append(reward)
                    ep_success.append(success)
        print(f"  Parsed {len(steps_log)} training metrics, {len(ep_steps)} episode logs")

In [ ]:
# Training curves
if HAS_WANDB:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes[0,0].plot(history["_step"], history["wm_loss"], alpha=0.3)
    axes[0,0].set_title("World Model Loss")
    axes[0,1].plot(history["_step"], history["actor_loss"], alpha=0.3, color="orange")
    axes[0,1].set_title("Actor Loss")
    axes[1,0].plot(history["_step"], history["episode_reward"], 'o', markersize=1, alpha=0.3)
    axes[1,0].set_title("Episode Reward")
    axes[1,1].plot(history["_step"], history["success"].rolling(50).mean(), color="green")
    axes[1,1].set_title("Success Rate (rolling 50)")
    for ax in axes.flat:
        ax.set_xlabel("Step")
    plt.suptitle("DreamerV3 Training — HM3D Easy (5M steps)")
    plt.tight_layout()
    plt.show()
elif 'steps_log' in dir() and steps_log:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes[0,0].plot(steps_log, wm_log, alpha=0.5)
    axes[0,0].set_title("World Model Loss")
    axes[0,1].plot(steps_log, actor_log, alpha=0.5, color="orange")
    axes[0,1].set_title("Actor Loss")
    if ep_steps:
        axes[1,0].plot(ep_steps, ep_rewards, 'o', markersize=1, alpha=0.3)
        axes[1,0].set_title("Episode Reward")
        # Rolling success rate
        window = min(50, len(ep_success))
        if window > 1:
            rolling_sr = np.convolve(ep_success, np.ones(window)/window, mode='valid')
            axes[1,1].plot(ep_steps[window-1:], rolling_sr, color="green")
        axes[1,1].set_title("Success Rate (rolling 50)")
    for ax in axes.flat:
        ax.set_xlabel("Step")
    plt.suptitle("DreamerV3 Training — HM3D Easy (from SLURM logs)")
    plt.tight_layout()
    plt.show()
else:
    print("No training data available to plot.")

In [ ]:
# Run or load eval results
EVAL_PATH = Path(RUN_DIR) / "eval_results.json" if RUN_DIR else None

if EVAL_PATH and EVAL_PATH.exists():
    with open(EVAL_PATH) as f:
        eval_results = json.load(f)
    print(f"Loaded {len(eval_results)} eval results from {EVAL_PATH}")
elif RUN_DIR:
    print(f"No eval results found. Run:")
    print(f"  uv run python -m src.dreamerv3.eval \\")
    print(f"    --checkpoint {RUN_DIR} \\")
    print(f"    --split val --max_geodesic 5.0 \\")
    print(f"    --num_episodes 200 --obs_size 64")
    eval_results = None
else:
    print("No run directory found.")
    eval_results = None

In [ ]:
# SR/SPL summary
if eval_results:
    sr = np.mean([r["success"] for r in eval_results])
    spl = np.mean([r["spl"] for r in eval_results])
    mean_r = np.mean([r["reward"] for r in eval_results])
    mean_steps = np.mean([r["steps"] for r in eval_results])

    print(f"{'Metric':<20} {'Value':>10}")
    print("-" * 32)
    print(f"{'Success Rate (SR)':<20} {sr:>10.3f}")
    print(f"{'SPL':<20} {spl:>10.3f}")
    print(f"{'Mean Reward':<20} {mean_r:>10.2f}")
    print(f"{'Mean Steps':<20} {mean_steps:>10.1f}")
    print(f"{'Episodes':<20} {len(eval_results):>10}")

    # SR by category
    from collections import defaultdict
    cat_results = defaultdict(list)
    for r in eval_results:
        cat_results[r.get("category", "unknown")].append(r["success"])

    print(f"\nSR by category:")
    cats = sorted(cat_results.keys())
    cat_srs = [np.mean(cat_results[c]) for c in cats]
    for c, s in zip(cats, cat_srs):
        print(f"  {c:<15} SR={s:.3f} (n={len(cat_results[c])})")

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(cats, cat_srs, color="#4C72B0")
    ax.bar_label(bars, fmt="{:.2f}", padding=3)
    ax.set_xlabel("Success Rate")
    ax.set_title("DreamerV3 SR by Object Category (Easy Episodes)")
    ax.set_xlim(0, max(cat_srs) * 1.3 if cat_srs else 1)
    plt.tight_layout()
    plt.show()
else:
    print("No eval results to analyze.")

In [ ]:
# Reward distribution
if eval_results:
    rewards = [r["reward"] for r in eval_results]
    successes = [r for r in eval_results if r["success"] > 0]
    failures = [r for r in eval_results if r["success"] == 0]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(rewards, bins=30, color="#4C72B0", edgecolor="white")
    axes[0].axvline(np.mean(rewards), color="red", linestyle="--", label=f"mean={np.mean(rewards):.2f}")
    axes[0].set_xlabel("Episode Reward")
    axes[0].set_title("Reward Distribution")
    axes[0].legend()

    step_counts = [r["steps"] for r in eval_results]
    axes[1].hist(step_counts, bins=30, color="#DD8452", edgecolor="white")
    axes[1].set_xlabel("Steps")
    axes[1].set_title("Episode Length Distribution")

    plt.suptitle(f"DreamerV3 Eval — {len(eval_results)} episodes, SR={sr:.3f}")
    plt.tight_layout()
    plt.show()
else:
    print("No eval results.")

In [ ]:
# Random baseline comparison
# Expected: random agent SR ≈ 0% on ObjectNav
RANDOM_EVAL_PATH = Path(RUN_DIR).parent / "random_eval.json" if RUN_DIR else None

if RANDOM_EVAL_PATH and RANDOM_EVAL_PATH.exists():
    with open(RANDOM_EVAL_PATH) as f:
        random_results = json.load(f)
elif eval_results:
    print("Random baseline not computed. To generate:")
    print("  Run eval with --checkpoint pointing to a freshly initialized (untrained) agent")
    random_results = None
else:
    random_results = None

if eval_results and random_results:
    dreamer_sr = np.mean([r["success"] for r in eval_results])
    random_sr = np.mean([r["success"] for r in random_results])

    fig, ax = plt.subplots(figsize=(5, 4))
    bars = ax.bar(["Random", "DreamerV3"], [random_sr, dreamer_sr],
                  color=["#95a5a6", "#4C72B0"])
    ax.bar_label(bars, fmt="{:.3f}")
    ax.set_ylabel("Success Rate")
    ax.set_title("DreamerV3 vs Random Baseline (Easy Episodes)")
    ax.set_ylim(0, max(dreamer_sr, random_sr) * 1.5 + 0.01)
    plt.tight_layout()
    plt.show()

## Conclusions

**Fill in after training completes:**

- **Did DreamerV3 learn?** [SR > 0 = yes]
- **Final SR/SPL:** [from eval above]
- **Which categories are easiest/hardest?** [from category breakdown]
- **Training stability:** [from loss curves]
- **Next steps:** Add 3D features (UNITE) and compare SR with/without